In [2]:
import pandas as pd

shows = pd.read_csv("titles.csv")
shows.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


In [3]:
# Filter
shows = shows[['id', 'title', 'description', 'genres']]

# Handle missing values
shows['description'] = shows['description'].fillna('')
shows['genres'] = shows['genres'].fillna('')

# Clean the string data
shows['genres'] = shows['genres'].str.replace('[', '', regex=False)
shows['genres'] = shows['genres'].str.replace(']', '', regex=False)
shows['genres'] = shows['genres'].str.replace("'", '', regex=False)
shows['genres'] = shows['genres'].str.replace(',', ' ', regex=False)

# Combine into a single text column
shows['tags'] = shows['description'] + " " + shows['genres']

# Display the final result
shows[['title', 'tags']].head()

,title,tags
0,Five Came Back: The Reference Films,This collection includes 12 World War II-era p...
1,Taxi Driver,A mentally unstable Vietnam War veteran works ...
2,Deliverance,Intent on seeing the Cahulawassee River before...
3,Monty Python and the Holy Grail,"King Arthur, accompanied by his squire, recrui..."
4,The Dirty Dozen,12 American military prisoners in World War II...


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Convert text into a numerical matrix
tfidf_matrix = tfidf.fit_transform(shows['tags'])

# Inspect the dimension of the matrix
tfidf_matrix.shape

(5850, 21063)

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the similarity scores for all 5,850 shows against each other
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a lookup table so we can easily find a show's row number by its title
indices = pd.Series(shows.index, index=shows['title']).drop_duplicates()

# Recommendation function
def get_recommendations(title, sim_matrix=cosine_sim):
    # Find the row number of the show the user typed in
    idx = indices[title]
    
    # Grab the similarity scores for that specific show against all others
    sim_scores = list(enumerate(sim_matrix[idx]))
    
    # Sort the list from highest match score to lowest
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Grab the top 5 matches (skipping index 0, because a show is a 100% match with itself)
    top_5_matches = sim_scores[1:6]
    
    # Extract just the row numbers for those top 5 matches
    show_indices = [i[0] for i in top_5_matches]
    
    # Return the actual titles from the original dataframe
    return shows['title'].iloc[show_indices]

5133    The Privilege
3675           Mortel
3165      The Society
1711             Dark
861        Dark Skies
Name: title, dtype: str

In [6]:
import numpy as np

def build_profile_recommendations(titles_list, sim_matrix=cosine_sim):
    # Find the row numbers for every valid show the user typed in
    idx_list = [indices[title] for title in titles_list if title in indices]
    
    if not idx_list:
        return "None of these shows were found in the dataset."
        
    # Grab the similarity scores for all selected shows and average them together
    # If a new show is 80% similar to title 1 and 60% similar to title 2, its profile score is 70%
    profile_scores = sim_matrix[idx_list].mean(axis=0)
    
    # Attach row numbers and sort from highest match to lowest
    sim_scores = list(enumerate(profile_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Extract the top 5 matches, skipping any shows the user already inputted
    recommended_indices = [i[0] for i in sim_scores if i[0] not in idx_list]
    top_5_matches = recommended_indices[:5]
    
    # Return the titles
    return shows['title'].iloc[top_5_matches]

# Test the profile builder with multiple shows
build_profile_recommendations(['Stranger Things', 'Gilmore Girls'])

1251    Gilmore Girls: A Year in the Life
1990                       No Escape Room
5133                        The Privilege
3165                          The Society
453                               Fashion
Name: title, dtype: str

In [7]:
import pickle

# Export the Pandas DataFrame containing the show titles and IDs
with open('shows_list.pkl', 'wb') as file:
    pickle.dump(shows, file)

# Export the massive pre-calculated math grid
with open('similarity_matrix.pkl', 'wb') as file:
    pickle.dump(cosine_sim, file)

print("Export complete! Check your VS Code file explorer.")

Export complete! Check your VS Code file explorer.
